# Six FFT-feature baselines: five-class ablations and eight-class LR/MLP

This notebook reads the same 56 compact `.pt` files and selects the same 55 `rank55` FFT features as the source notebook. It evaluates six baselines using one consistent train/test protocol.

| configuration | task | feature count |
|---|---:|---:|
| All three FFT projections | Five-class LR | 55 |
| Without FFT{\|x\|²} | Five-class LR | 44 |
| Without FFT{x²} | Five-class LR | 33 |
| Without FFT{x⁴} | Five-class LR | 33 |
| 55-D FFT–LR | Eight-class LR | 55 |
| 55-D FFT–MLP | Eight-class MLP | 55 |

The five-class task maps `QPSK`, `16QAM`, `64QAM`, and `256QAM` to `QPSK-like`; `BPSK`, `MSK`, `DQPSK`, and `8PSK` remain separate. The eight-class task keeps all eight original signal types separate.

For both tasks, `2018train20` uses 2018 batches 1–20 and `2022train20` uses 2022 batches 1–20. A same-year test uses batches 25–28; a cross-year test uses all batches 1–28 of the other year. Each baseline therefore produces four accuracies, for a final 6×4 (24-value) table. No confusion matrices are created.

Nothing is saved by the code. Models, predictions, and accuracy values remain in memory and are displayed only in this notebook.

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Run from the directory containing the 2018/ and 2022/ folders.
PT_ROOT = Path.cwd()
YEARS = ("2018", "2022")
TRAIN_LAST_BATCH = 20
RANDOM_STATE = 17

LABEL8_ORDER = ["BPSK", "MSK", "DQPSK", "QPSK", "8PSK", "16QAM", "64QAM", "256QAM"]
LABEL5_ORDER = ["BPSK", "MSK", "DQPSK", "8PSK", "QPSK-like"]

RANK55_FEATURES = [
    "x4_top1_minus_top2_db",
    "x4_top2_db",
    "x4_top3_db",
    "x4_top1_af",
    "x4_top1_db",
    "x4_int_grid_3rd_db",
    "x4_top1_minus_top5_db",
    "x4_top4_db",
    "x2_nonzero_count_gt_m10",
    "x2_nonzero_count_gt_m20",
    "x4_int_grid_2nd_db",
    "x2_top4_db",
    "x2_top1_minus_top5_db",
    "x2_top3_db",
    "x2_half_grid_best_db",
    "x2_half_grid_2nd_db",
    "x4_nonzero_count_gt_m10",
    "x2_top1_db",
    "abs_x2_top4_db",
    "abs_x2_top1_minus_top5_db",
    "x4_int_grid_count_gt_m10",
    "x2_top1_minus_top2_db",
    "x2_top2_db",
    "x4_top2_af",
    "x4_half_grid_count_gt_m10",
    "x2_half_grid_count_gt_m20",
    "x2_half_grid_count_gt_m30",
    "x2_top1_af",
    "abs_x2_nonzero_count_gt_m30",
    "x4_top1_af_over_rs",
    "x2_top3_af",
    "x2_int_grid_count_gt_m20",
    "x2_int_grid_2nd_db",
    "x2_top2_af",
    "abs_x2_nonzero_count_gt_m20",
    "abs_x2_top1_minus_top2_db",
    "abs_x2_nonzero_best_db",
    "abs_x2_top2_db",
    "x2_int_grid_count_gt_m30",
    "x2_half_grid_count_gt_m10",
    "x4_top3_af",
    "abs_x2_top3_db",
    "x4_half_grid_3rd_db",
    "x2_nonzero_count_gt_m30",
    "x4_nonzero_count_gt_m20",
    "x2_nonzero_best_db",
    "abs_x2_top4_af",
    "x4_half_grid_count_gt_m20",
    "x2_half_grid_3rd_db",
    "x4_top4_af",
    "x4_half_minus_int_top3_db",
    "x4_half_grid_best_db",
    "abs_x2_half_grid_mean_db",
    "abs_x2_int_grid_mean_db",
    "x4_half_grid_mean_top3_db",
]
assert len(RANK55_FEATURES) == len(set(RANK55_FEATURES)) == 55

In [2]:
FEATURE_SETS = {
    "all55": RANK55_FEATURES,
    "without_abs_x2": [name for name in RANK55_FEATURES if not name.startswith("abs_x2_")],
    "without_x2": [name for name in RANK55_FEATURES if not name.startswith("x2_")],
    "without_x4": [name for name in RANK55_FEATURES if not name.startswith("x4_")],
}
EXPECTED_FEATURE_COUNTS = {"all55": 55, "without_abs_x2": 44, "without_x2": 33, "without_x4": 33}
assert {key: len(value) for key, value in FEATURE_SETS.items()} == EXPECTED_FEATURE_COUNTS

BASELINES = [
    {"key": "all_fft_5lr", "configuration": "All three FFT projections", "task": "Five-class LR", "feature_set": "all55", "target": "five", "model": "lr"},
    {"key": "no_abs_x2_5lr", "configuration": "Without FFT{|x|2}", "task": "Five-class LR", "feature_set": "without_abs_x2", "target": "five", "model": "lr"},
    {"key": "no_x2_5lr", "configuration": "Without FFT{x2}", "task": "Five-class LR", "feature_set": "without_x2", "target": "five", "model": "lr"},
    {"key": "no_x4_5lr", "configuration": "Without FFT{x4}", "task": "Five-class LR", "feature_set": "without_x4", "target": "five", "model": "lr"},
    {"key": "fft55_8lr", "configuration": "55-D FFT–LR", "task": "Eight-class LR", "feature_set": "all55", "target": "eight", "model": "lr"},
    {"key": "fft55_8mlp", "configuration": "55-D FFT–MLP", "task": "Eight-class MLP", "feature_set": "all55", "target": "eight", "model": "mlp"},
]
assert len(BASELINES) == 6

display(pd.DataFrame([
    {
        "configuration": item["configuration"],
        "task": item["task"],
        "feature_count": len(FEATURE_SETS[item["feature_set"]]),
    }
    for item in BASELINES
]))

,configuration,task,feature_count
0,All three FFT projections,Five-class LR,55
1,Without FFT{|x|2},Five-class LR,44
2,Without FFT{x2},Five-class LR,33
3,Without FFT{x4},Five-class LR,33
4,55-D FFT–LR,Eight-class LR,55
5,55-D FFT–MLP,Eight-class MLP,55


In [3]:
def select_features_by_name(X, source_names, selected_names):
    source_names = list(source_names)
    if len(source_names) != len(set(source_names)):
        raise ValueError("Source feature names must be unique")
    source_index = {name: index for index, name in enumerate(source_names)}
    missing = [name for name in selected_names if name not in source_index]
    if missing:
        raise KeyError(f"Requested rank55 features are missing: {missing}")
    return np.asarray(X)[:, [source_index[name] for name in selected_names]]


def label5_from_signal_type(signal_type):
    signal_type = str(signal_type).upper()
    if signal_type in {"BPSK", "MSK", "DQPSK", "8PSK"}:
        return signal_type
    if signal_type in {"QPSK", "16QAM", "64QAM", "256QAM"}:
        return "QPSK-like"
    raise ValueError(f"Unsupported signal_type={signal_type!r}")


def encode_labels(labels, label_order):
    label_to_index = {label: index for index, label in enumerate(label_order)}
    encoded = pd.Series(labels).map(label_to_index)
    if encoded.isna().any():
        raise ValueError(f"Unexpected labels: {sorted(set(labels) - set(label_order))}")
    return encoded.to_numpy(dtype=np.int64)


def load_rank55_vectors(pt_root, years=YEARS):
    pt_root = Path(pt_root)
    metadata_parts = []
    feature_parts = []
    source_feature_names = None
    loaded_file_count = 0
    t0 = time.time()

    for year in years:
        files = sorted((pt_root / year).glob("*_boi_shift_fft3_compact_vectors.pt"))
        if len(files) != 28:
            raise FileNotFoundError(
                f"Expected 28 compact-vector .pt files under {pt_root / year}, found {len(files)}"
            )

        for file_index, path in enumerate(files, start=1):
            pack = torch.load(path, map_location="cpu", weights_only=False)
            samples = pack["samples"]
            current_names = list(pack["vector_feature_names"]["lr445"])
            if source_feature_names is None:
                source_feature_names = current_names
            elif source_feature_names != current_names:
                raise ValueError(f"lr445 feature-name mismatch in {path}")

            lr445 = np.vstack([
                np.asarray(sample["vectors"]["lr445"], dtype=np.float32)
                for sample in samples
            ])
            feature_parts.append(
                select_features_by_name(lr445, current_names, RANK55_FEATURES).astype(np.float32, copy=False)
            )

            rows = []
            for sample in samples:
                meta = sample["meta"]
                signal_type = str(meta["signal_type"]).upper()
                if signal_type not in LABEL8_ORDER:
                    raise ValueError(f"Unsupported signal_type={signal_type!r} in {path}")
                rows.append({
                    "year": year,
                    "batch_id": int(meta.get("batch_id", pack.get("batch_id", -1))),
                    "signal_type": signal_type,
                    "pt_name": path.name,
                })
            metadata_parts.append(pd.DataFrame(rows))
            loaded_file_count += 1
            print(
                f"{year} {file_index:02d}/{len(files):02d} {path.name}: "
                f"rows={len(samples)}, elapsed={time.time() - t0:.1f}s",
                flush=True,
            )
            del pack, samples, lr445

    metadata = pd.concat(metadata_parts, ignore_index=True)
    X55 = np.vstack(feature_parts).astype(np.float32, copy=False)
    X55[~np.isfinite(X55)] = np.nan
    assert loaded_file_count == 56
    assert X55.shape == (len(metadata), 55)
    assert set(metadata["signal_type"].unique()) == set(LABEL8_ORDER)
    return metadata, X55


meta_df, X55 = load_rank55_vectors(PT_ROOT)
rank55_index = {name: index for index, name in enumerate(RANK55_FEATURES)}
X_BY_FEATURE_SET = {
    key: X55[:, [rank55_index[name] for name in names]]
    for key, names in FEATURE_SETS.items()
}

y8_labels = meta_df["signal_type"].to_numpy()
y5_labels = np.array([label5_from_signal_type(label) for label in y8_labels])
Y_BY_TARGET = {
    "five": encode_labels(y5_labels, LABEL5_ORDER),
    "eight": encode_labels(y8_labels, LABEL8_ORDER),
}
assert set(np.unique(Y_BY_TARGET["five"])) == set(range(5))
assert set(np.unique(Y_BY_TARGET["eight"])) == set(range(8))
print(f"Loaded {len(meta_df):,} signals from 56 .pt files; X55 shape={X55.shape}")

2018 01/28 2018_batch_01_boi_shift_fft3_compact_vectors.pt: rows=3992, elapsed=11.4s
2018 02/28 2018_batch_02_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=22.1s
2018 03/28 2018_batch_03_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=32.7s
2018 04/28 2018_batch_04_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=43.1s
2018 05/28 2018_batch_05_boi_shift_fft3_compact_vectors.pt: rows=3992, elapsed=53.3s
2018 06/28 2018_batch_06_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=63.4s
2018 07/28 2018_batch_07_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=73.3s
2018 08/28 2018_batch_08_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=83.2s
2018 09/28 2018_batch_09_boi_shift_fft3_compact_vectors.pt: rows=3992, elapsed=93.0s
2018 10/28 2018_batch_10_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=103.0s
2018 11/28 2018_batch_11_boi_shift_fft3_compact_vectors.pt: rows=3984, elapsed=112.8s
2018 12/28 2018_batch_12_boi_shift_fft3_compact_vectors.pt: row

In [4]:
def make_train_mask(train_year):
    return (
        meta_df["year"].eq(str(train_year))
        & meta_df["batch_id"].between(1, TRAIN_LAST_BATCH)
    ).to_numpy()


def make_test_mask(train_year, test_year):
    year_mask = meta_df["year"].eq(str(test_year))
    if str(train_year) == str(test_year):
        batch_mask = meta_df["batch_id"].between(25, 28)
    else:
        batch_mask = meta_df["batch_id"].between(1, 28)
    return (year_mask & batch_mask).to_numpy()


def make_lr():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            C=3,
            class_weight="balanced",
            max_iter=1000,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


def make_mlp():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", MLPClassifier(
            hidden_layer_sizes=(64,),
            activation="relu",
            solver="adam",
            batch_size=512,
            learning_rate_init=1e-3,
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=RANDOM_STATE,
            verbose=True,
        )),
    ])


evaluation_plan = []
for train_year in YEARS:
    train_mask = make_train_mask(train_year)
    for test_year in YEARS:
        test_mask = make_test_mask(train_year, test_year)
        if np.any(train_mask & test_mask):
            raise RuntimeError(f"Train/test overlap: train={train_year}, test={test_year}")
        test_batches = "25-28" if train_year == test_year else "1-28"
        evaluation_plan.append({
            "training": f"{train_year}train20",
            "test": f"{test_year} batch {test_batches}",
            "train_n": int(train_mask.sum()),
            "test_n": int(test_mask.sum()),
        })
display(pd.DataFrame(evaluation_plan))

,training,test,train_n,test_n
0,2018train20,2018 batch 25-28,79720,15960
1,2018train20,2022 batch 1-28,79720,112000
2,2022train20,2018 batch 1-28,80000,111632
3,2022train20,2022 batch 25-28,80000,16000


In [5]:
result_rows = []
trained_models = {}

for baseline_index, baseline in enumerate(BASELINES, start=1):
    X_baseline = X_BY_FEATURE_SET[baseline["feature_set"]]
    y = Y_BY_TARGET[baseline["target"]]
    print("\n" + "=" * 100)
    print(
        f"Baseline {baseline_index}/6: {baseline['configuration']} | {baseline['task']} | "
        f"features={X_baseline.shape[1]}"
    )

    for train_year in YEARS:
        train_name = f"{train_year}train20"
        train_mask = make_train_mask(train_year)
        expected_classes = 5 if baseline["target"] == "five" else 8
        if set(np.unique(y[train_mask])) != set(range(expected_classes)):
            raise RuntimeError(f"{train_name} does not contain every target class")

        model = make_lr() if baseline["model"] == "lr" else make_mlp()
        print(f"Training {train_name}: n={train_mask.sum():,}")
        t0 = time.time()
        model.fit(X_baseline[train_mask], y[train_mask])
        fit_seconds = time.time() - t0
        trained_models[(baseline["key"], train_name)] = model

        if baseline["model"] == "mlp":
            weight_shapes = [weights.shape for weights in model.named_steps["classifier"].coefs_]
            assert weight_shapes == [(55, 64), (64, 8)], weight_shapes

        for test_year in YEARS:
            test_batches = "25-28" if train_year == test_year else "1-28"
            test_mask = make_test_mask(train_year, test_year)
            y_pred = model.predict(X_baseline[test_mask])
            acc = accuracy_score(y[test_mask], y_pred)
            evaluation = f"{train_name} → {test_year} b{test_batches}"
            result_rows.append({
                "baseline_key": baseline["key"],
                "configuration": baseline["configuration"],
                "task": baseline["task"],
                "feature_count": int(X_baseline.shape[1]),
                "evaluation": evaluation,
                "accuracy": float(acc),
                "accuracy_percent": float(100 * acc),
                "fit_seconds": float(fit_seconds),
            })
            print(f"  {evaluation}: accuracy={100 * acc:.4f}%")

results = pd.DataFrame(result_rows)
assert len(results) == 6 * 4


Baseline 1/6: All three FFT projections | Five-class LR | features=55
Training 2018train20: n=79,720
  2018train20 → 2018 b25-28: accuracy=99.9561%
  2018train20 → 2022 b1-28: accuracy=99.9000%
Training 2022train20: n=80,000
  2022train20 → 2018 b1-28: accuracy=99.8755%
  2022train20 → 2022 b25-28: accuracy=99.9938%

Baseline 2/6: Without FFT{|x|2} | Five-class LR | features=44
Training 2018train20: n=79,720
  2018train20 → 2018 b25-28: accuracy=99.9311%
  2018train20 → 2022 b1-28: accuracy=99.9214%
Training 2022train20: n=80,000
  2022train20 → 2018 b1-28: accuracy=99.7984%
  2022train20 → 2022 b25-28: accuracy=99.9688%

Baseline 3/6: Without FFT{x2} | Five-class LR | features=33
Training 2018train20: n=79,720
  2018train20 → 2018 b25-28: accuracy=98.6529%
  2018train20 → 2022 b1-28: accuracy=90.0902%
Training 2022train20: n=80,000
  2022train20 → 2018 b1-28: accuracy=97.8743%
  2022train20 → 2022 b25-28: accuracy=99.3750%

Baseline 4/6: Without FFT{x4} | Five-class LR | features=33


In [6]:
evaluation_order = [
    "2018train20 → 2018 b25-28",
    "2018train20 → 2022 b1-28",
    "2022train20 → 2018 b1-28",
    "2022train20 → 2022 b25-28",
]
baseline_order = [item["key"] for item in BASELINES]

accuracy_table = (
    results.pivot(index="baseline_key", columns="evaluation", values="accuracy_percent")
    .reindex(index=baseline_order, columns=evaluation_order)
)
accuracy_table.index = [
    f"{item['configuration']} | {item['task']} | {len(FEATURE_SETS[item['feature_set']])}D"
    for item in BASELINES
]
accuracy_table.index.name = "configuration | task | dimensions"
assert accuracy_table.shape == (6, 4)
display(accuracy_table.style.format("{:.4f}%").set_caption("Accuracy summary: 6 baselines × 4 evaluations"))

evaluation,2018train20 → 2018 b25-28,2018train20 → 2022 b1-28,2022train20 → 2018 b1-28,2022train20 → 2022 b25-28
configuration | task | dimensions,,,,
All three FFT projections | Five-class LR | 55D,99.9561%,99.9000%,99.8755%,99.9938%
Without FFT{|x|2} | Five-class LR | 44D,99.9311%,99.9214%,99.7984%,99.9688%
Without FFT{x2} | Five-class LR | 33D,98.6529%,90.0902%,97.8743%,99.3750%
Without FFT{x4} | Five-class LR | 33D,71.5288%,69.6643%,70.5058%,72.1375%
55-D FFT–LR | Eight-class LR | 55D,83.9599%,77.5518%,81.3485%,83.2062%
55-D FFT–MLP | Eight-class MLP | 55D,84.5489%,81.2179%,83.4886%,85.3000%


## Output

The final table contains exactly 24 accuracy values: six baselines by four evaluations. No confusion matrix, checkpoint, prediction file, or result file is generated.